# Notebook 08: Probing Classifiers

## Why This Matters
A linear probe answers a precise question: *does the representation encode X?* If a linear classifier trained on frozen features can predict X, then X is linearly decodable from the representation — the network has organized that information into a linearly separable structure.

This matters for two reasons. First, it's the standard evaluation protocol for self-supervised learning — fairer than fine-tuning, which can obscure representation quality. Second, probing at different layers reveals *where* information lives in the network, which matters for feature extraction, transfer learning, and understanding what the model has actually learned.

## Structure
1. Linear probing protocol — why frozen features and a linear classifier (narrative)
2. Layer-by-layer probing of CLIP
3. Layer-by-layer probing of DINO
4. Probing for different properties: class, color, shape, texture
5. When to probe vs when to fine-tune
6. Bridge to embedding evaluation

## What You'll Learn
- Why linear probing is a cleaner measure of representation quality than fine-tuning
- How semantic information builds up across transformer layers
- How to probe frozen features for specific properties
- When probing results predict downstream task performance (and when they don't)


## Background

### Why linear probing?

Fine-tuning is powerful but confounding: a model with poor representations can still achieve high accuracy on a downstream task if the fine-tuning process restructures the entire network. Linear probing isolates the question "what does the representation already encode" by allowing only a linear transformation.

If your frozen features score 80% with a linear classifier, the encoder has organized semantics into linearly separable structure. That's a strong signal about representation quality that fine-tuning accuracy doesn't reveal.

### The protocol

```
1. Freeze encoder weights entirely
2. Extract features from a specific layer on train and test sets
3. Train a linear classifier (logistic regression or linear SVM) on train features
4. Report test accuracy
5. Repeat at each layer to build a layer-by-layer profile
```

The classifier must be fast to train — use scikit-learn LogisticRegression with L-BFGS, not SGD, to avoid confounding the representation quality with optimizer quality.

### What probing reveals

Early layers in transformers encode local, low-level features: edges, textures, colors. Later layers encode semantic, high-level features: object identity, scene type, relationships. The transition happens at different depths depending on the model size and pretraining objective.

In CLIP, semantic information (class identity) peaks near the final layers because the objective is global (image-caption matching). In DINO, spatial information (location, local texture) remains strong throughout because the self-distillation objective preserves local information through the patch tokens.


In [ ]:
# %pip install -q torch torchvision openai-clip numpy matplotlib scikit-learn

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import clip
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()
print("CLIP ViT-B/32 loaded")

dino_model = torch.hub.load("facebookresearch/dino:main", "dino_vits8", pretrained=True)
dino_model = dino_model.to(device).eval()
print("DINO ViT-S/8 loaded")

classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

## 1. Layer-by-Layer Feature Extraction

In [ ]:
def extract_layer_features_clip(model, loader, layer_idx: int, device: str):
    """Extract features from a specific transformer block in CLIP's image encoder."""
    features, labels = [], []
    hooks = []

    captured = {}
    def make_hook(idx):
        def hook_fn(module, input, output):
            if idx == layer_idx:
                captured['feat'] = output.detach()
        return hook_fn

    for i, block in enumerate(model.visual.transformer.resblocks):
        h = block.register_forward_hook(make_hook(i))
        hooks.append(h)

    with torch.no_grad():
        for imgs, lbls in loader:
            captured.clear()
            imgs = imgs.to(device)
            _ = model.encode_image(imgs)
            # CLS token is position 0 in ViT sequence
            feat = captured['feat'][:, 0, :].cpu().numpy()  # [B, D]
            features.append(feat); labels.append(lbls.numpy())

    for h in hooks: h.remove()
    return np.concatenate(features), np.concatenate(labels)


# Data
clip_train = torchvision.datasets.CIFAR10("/tmp/cifar10", train=True,  download=True,  transform=clip_preprocess)
clip_test  = torchvision.datasets.CIFAR10("/tmp/cifar10", train=False, download=False, transform=clip_preprocess)
clip_train_loader = DataLoader(clip_train, batch_size=256, shuffle=False, num_workers=0)
clip_test_loader  = DataLoader(clip_test,  batch_size=256, shuffle=False, num_workers=0)

n_layers = len(clip_model.visual.transformer.resblocks)
print(f"CLIP ViT-B/32 has {n_layers} transformer blocks")
print("Probing every other layer...")

clip_accs = {}
probe_layers = list(range(0, n_layers, 2)) + [n_layers - 1]
for layer in sorted(set(probe_layers)):
    train_f, train_l = extract_layer_features_clip(clip_model, clip_train_loader, layer, device)
    test_f,  test_l  = extract_layer_features_clip(clip_model, clip_test_loader,  layer, device)
    sc = StandardScaler()
    clf = LogisticRegression(max_iter=300, C=0.1, random_state=42, n_jobs=-1)
    clf.fit(sc.fit_transform(train_f), train_l)
    acc = accuracy_score(test_l, clf.predict(sc.transform(test_f)))
    clip_accs[layer] = acc
    print(f"  Layer {layer:2d}: {acc:.3f}")

In [ ]:
def extract_layer_features_dino(model, loader, layer_idx: int, device: str):
    """Extract CLS token features from a specific DINO block."""
    features, labels = [], []
    captured = {}

    def hook_fn(module, input, output):
        captured['feat'] = output.detach()

    hook = model.blocks[layer_idx].register_forward_hook(hook_fn)

    with torch.no_grad():
        for imgs, lbls in loader:
            captured.clear()
            imgs = imgs.to(device)
            _ = model(imgs)
            feat = captured['feat'][:, 0, :].cpu().numpy()
            features.append(feat); labels.append(lbls.numpy())

    hook.remove()
    return np.concatenate(features), np.concatenate(labels)


dino_transform = T.Compose([T.Resize(224), T.CenterCrop(224), T.ToTensor(),
                             T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
dino_train = torchvision.datasets.CIFAR10("/tmp/cifar10", train=True,  download=False, transform=dino_transform)
dino_test  = torchvision.datasets.CIFAR10("/tmp/cifar10", train=False, download=False, transform=dino_transform)
dino_train_loader = DataLoader(dino_train, batch_size=256, shuffle=False, num_workers=0)
dino_test_loader  = DataLoader(dino_test,  batch_size=256, shuffle=False, num_workers=0)

n_dino_layers = len(dino_model.blocks)
print(f"\nDINO ViT-S/8 has {n_dino_layers} transformer blocks")
print("Probing every other layer...")

dino_accs = {}
for layer in range(0, n_dino_layers, 2):
    train_f, train_l = extract_layer_features_dino(dino_model, dino_train_loader, layer, device)
    test_f,  test_l  = extract_layer_features_dino(dino_model, dino_test_loader,  layer, device)
    sc = StandardScaler()
    clf = LogisticRegression(max_iter=300, C=0.1, random_state=42, n_jobs=-1)
    clf.fit(sc.fit_transform(train_f), train_l)
    acc = accuracy_score(test_l, clf.predict(sc.transform(test_f)))
    dino_accs[layer] = acc
    print(f"  Layer {layer:2d}: {acc:.3f}")

## 2. Layer-by-Layer Accuracy Curve

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

clip_layers = sorted(clip_accs.keys())
dino_layers = sorted(dino_accs.keys())

ax.plot([l/(n_layers-1) for l in clip_layers],
        [clip_accs[l] for l in clip_layers],
        marker='o', label='CLIP ViT-B/32', color='royalblue')
ax.plot([l/(n_dino_layers-1) for l in dino_layers],
        [dino_accs[l] for l in dino_layers],
        marker='s', label='DINO ViT-S/8', color='darkorange')

ax.set_xlabel("Relative layer depth (0=first, 1=last)")
ax.set_ylabel("Linear probe accuracy (CIFAR-10)")
ax.set_title("Where does semantic information live?")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("Key observations:")
print("  - Early layers: low accuracy — local, low-level features")
print("  - Later layers: high accuracy — semantic, class-level features")
print("  - Final layer is often best for classification but not always for transfer")
print("  - CLIP and DINO may peak at different relative depths")

## 3. What the Linear Probe Actually Learns

In [ ]:
# Train a probe on final CLIP features and examine which classes are hardest
clip_train_f, clip_train_l = extract_layer_features_clip(clip_model, clip_train_loader, n_layers-1, device)
clip_test_f,  clip_test_l  = extract_layer_features_clip(clip_model, clip_test_loader,  n_layers-1, device)

sc = StandardScaler()
clf = LogisticRegression(max_iter=300, C=0.1, random_state=42, n_jobs=-1)
clf.fit(sc.fit_transform(clip_train_f), clip_train_l)
preds = clf.predict(sc.transform(clip_test_f))

print("Per-class accuracy from linear probe on CLIP final layer:")
print("-" * 40)
for i, cls in enumerate(classes):
    mask = clip_test_l == i
    acc = (preds[mask] == clip_test_l[mask]).mean()
    bar = '█' * int(acc * 20)
    print(f"  {cls:12s}: {acc:.2f} {bar}")

# Confusion: which classes get mixed up?
print("\nTop confusions (most frequent misclassifications):")
confusions = {}
for true, pred in zip(clip_test_l, preds):
    if true != pred:
        key = (classes[true], classes[pred])
        confusions[key] = confusions.get(key, 0) + 1
for (true_cls, pred_cls), count in sorted(confusions.items(), key=lambda x:-x[1])[:8]:
    print(f"  {true_cls:12s} → {pred_cls:12s}: {count} times")

## 4. When to Probe vs When to Fine-Tune

**Use linear probing when:**
- You want to measure representation quality without confounding it with adaptation
- You have very little labeled data (< 1000 examples)
- You need interpretability: which layers encode which properties
- You're comparing multiple pretrained models fairly

**Use fine-tuning when:**
- Representation quality is already established and you want max performance
- You have enough labeled data (> 10k examples) to adapt meaningfully
- The distribution gap between pretraining and your domain is large
- You have compute budget for full fine-tuning or LoRA

**Practical rule:** probe first to understand what the model already knows, then fine-tune only what's needed. If linear probe accuracy is already good enough, skip fine-tuning entirely.


## 5. Bridge to Embedding Evaluation

Linear probing is one evaluation protocol for representations. But it only measures class accuracy — it doesn't tell you about:

- **Calibration:** are similarity scores actually meaningful as probabilities?
- **Retrieval quality:** precision@k, recall@k, NDCG
- **Clustering:** do similar classes cluster without supervision?
- **Cross-dataset transfer:** does accuracy on CIFAR predict accuracy on other datasets?

Notebook 09 covers the full embedding evaluation toolkit: STS for text, recall@k for retrieval, MTEB for multilingual evaluation, and the pitfalls of treating any single benchmark as ground truth.


## Summary

| Concept | Key Point |
|---------|-----------|
| Linear probe | Frozen encoder + linear classifier = "does this representation encode X?" |
| Protocol | Freeze weights → extract features → train linear layer → report test accuracy |
| Layer analysis | Early layers: local features; later layers: semantic features |
| Vs fine-tuning | Probing isolates representation quality; fine-tuning conflates representation + adaptation |
| CLIP vs DINO | Both peak at final layers for classification; DINO retains more spatial signal |

**Next:** Notebook 09 — Embedding Evaluation. The full toolkit: STS benchmarks for text, recall@k for retrieval, MTEB for cross-lingual evaluation, and how to pick the right metric for your task.
